# 第13回　学習曲線とハイパーパラメータ
***
> **前提**: 第6回の交差検証を発展させ，過学習の判断方法を学びます。

> ⚠️ **この課題で身につけること：コーディングではなく「AI（機械学習）の中身の理解」です。**
>
> コードは AI に書かせても構いません。重要なのは「**なぜその処理を選ぶのか**」「**パラメータや特徴量を変えると結果がどう変わるのか**」を理解し、提出物で示すことです。各問には学習目標を示すタグが付いています。
>
> | タグ | 意味 | あなたがすること |
> |---|---|---|
> | **【骨格】** | 動くコードは与えられている | 設計上の決定点（数値・選択肢・特徴量）だけを変更する |
> | **【選択】** | 適切な手法を選ぶ問題 | 複数候補から選び、**理由**を解答用コードセルに書く |
> | **【実験】** | 試行錯誤の記録 | パラメータ等を変えて結果を表に記録し、**考察**する |
> | **【説明】** | 理解の証跡 | 与えられたコードの各行に `# 説明:` で意味を書く |
>
> コードは原則として完成形ですが、**各問の「核心となる最低限の数行」は `# ★あなたが書く★` として空欄**にしてあります。AI に頼り切らず、要となる処理は自分で書けることも確認します（ボイラープレートは提供済み）。
>
> 各問の **✍️ 解答用コードセル**（`# (1-a)` 形式の変数・文字列）に、設計判断・理由・実験結果・考察を**項目ごとに**記入してください。これが採点対象です。

## 目次
1. learning_curve
2. validation_curve
3. 交差検証の復習
4. 過学習の判断

---

## この回で学ぶこと

### バイアスとバリアンスのトレードオフ

機械学習モデルの「誤差」は2種類に分解できる：

```
汎化誤差 = バイアス² + バリアンス + ノイズ

【高バイアス（Underfitting）】
  モデルが単純すぎ → 訓練データにすら当てはまらない
  → 訓練スコア低，テストスコア低
  → 解決策：モデルを複雑にする（max_depth を増やすなど）

【高バリアンス（Overfitting）】
  モデルが複雑すぎ → 訓練データは完璧だが，テストデータは失敗
  → 訓練スコア高，テストスコア低（大きく差がある）
  → 解決策：正則化，データを増やす，モデルを単純にする
```

### 学習曲線（Learning Curve）

**訓練データのサイズを変えながら，訓練スコアとテストスコアの変化をプロットしたもの**。

```
【良い学習曲線のパターン】
訓練スコア  ↓  ̄ ̄ ̄\        ← データが増えると訓練スコアは少し下がる
検証スコア  /          ← データが増えると検証スコアは上がる
           両者が収束 → underfittingもoverfittingもない

【過学習の場合】
訓練スコア  ̄ ̄ ̄ ̄ ̄ ̄（高いまま）
検証スコア  ___________（低いまま，差が大きい）
```

- データが少ない時に大きな差がある：正則化またはデータ収集が必要
- データが多くなっても差が縮まらない：モデルが複雑すぎ

### 検証曲線（Validation Curve）

**ハイパーパラメータを変化させながら，訓練スコアと検証スコアをプロット**。最適なハイパーパラメータを視覚的に選ぶための手法だ。

```
【決定木の max_depth の場合】
max_depth が小さい：高バイアス（underfitting）
max_depth が大きい：高バリアンス（overfitting）
最適 max_depth：検証スコアが最大になる点
```

### 交差検証（Cross-Validation）の重要性

単一の train/test 分割では，その分割方法に結果が依存してしまう（運次第）。**k-fold 交差検証** はデータをk個に分割し，各分割でテストデータを入れ替えてk回評価することで，より安定した評価ができる。

> **卒業研究での指針**: 論文でハイパーパラメータを報告する際は，必ず「どのデータで選んだか（交差検証 or テストデータ）」を明記すること。テストデータでハイパーパラメータを選ぶと**テストデータのリーク**になる。

---

## 📚 将来の自分のためのメモ — 大学研究での応用


### どんな研究で使われるか

| 分野 | 具体例 |
|---|---|
| 全分野の機械学習研究 | モデル選択，過学習の診断，「データをあと何件集めればよいか」の判断 |
| 医学・生物学 | サンプル数が限られる研究での学習曲線分析 |
| 工学 | センサーデータのモデル複雑度チューニング（`max_depth` など） |
| 論文査読 | 学習曲線・検証曲線がないと「過学習を確認していない」と指摘されやすい |

### 応用できる場面

- **過学習か未学習かを判断**したい（訓練スコアと検証スコアのギャップ）
- **ハイパーパラメータの最適値**を体系的に探したい（検証曲線，GridSearchCV）
- **データ収集の費用対効果**を検討（学習曲線がまだ上昇中なら，データ追加が有効かも）
- 卒論・論文で**実験設計の説得力**を高めたい

### 応用しにくい・向かない場面

- **大規模事前学習モデル**（BERT，GPT など）のファインチューニング— 評価の枠組みが異なる（エポック数，学習率スケジュールなど）
- **オンライン学習・継続学習**（データが順次届く）— 静的な learning_curve とは別設計
- 検証曲線の**1点だけ**を見て結論づける（曲線全体の形状が重要）
- テストデータでハイパーパラメータを選ぶ（**リーク**— 本回で強調した最重要禁忌）

### 研究で報告するときのポイント

1. **学習曲線**と**検証曲線**（または validation curve）の図を論文に載せる
2. ハイパーパラメータは**交差検証**で選び，最終性能だけを**未使用のテストデータ**で報告
3. 「高バイアス / 高バリアンス」のどちらかを言語で説明し，**対策**（正則化，データ追加，モデル簡素化）を述べる
4. `GridSearchCV` の探索範囲と評価指標を Methods に書く

### 関連する発展トピック（調べてみると良い）

- Nested cross-validation（ハイパーパラメータ選択のバイアスをさらに抑える）
- Optuna / Bayesian optimization（探索の効率化）
- Early stopping（深層学習での過学習防止— 第15回以降と連動）

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import learning_curve, validation_curve, train_test_split
from sklearn.tree import DecisionTreeClassifier


## 問題1　学習曲線の描画　【説明】
***

### `cv=5` の意味

`cv=5` は5分割交差検証を意味する。各訓練サイズで：
1. データを5つに分割
2. 4つを訓練，1つを検証に使って評価
3. これを5回繰り返し，5つのスコアの平均と標準偏差を算出

`train_scores` の shape は `(train_sizes の個数, cv=5)` になる。そのため `np.mean(train_scores, axis=1)` で各サイズの平均を取る。

### 標準偏差の帯を描く方法

各点での不確かさ（ばらつき）を帯として描くと，グラフの信頼性が上がる：

```python
plt.fill_between(
    train_sizes,
    train_mean - train_std,
    train_mean + train_std,
    alpha=0.2
)
```

論文やレポートではこのような信頼区間を付けることが推奨される。

### 課題

下のコードセルは，乳がんデータで `learning_curve` を計算して学習曲線を描画する **完成形**です。今回は自分でコードを書くのではなく，**`learning_curve` が何を返し，訓練スコアと検証スコアの差が何を意味するのかを読み解く**のが目的です。

**各行の `# 説明:` の右に，その行が何をしているかを自分の言葉で書いて**ください（コード自体は変更しないこと）。書き終えたらセルを実行し，学習曲線を観察してください。

説明を書くときは，次の問いを意識してください：

- `learning_curve` が返す `train_scores` と `valid_scores` は，それぞれ何を測ったスコアか？
- `train_scores` の shape はなぜ `(訓練サイズ数, cv数)` なのか？ なぜ `axis=1` 方向に平均を取るのか？
- 訓練スコアと検証スコアの **差** が大きいことは，モデルの状態について何を意味するのか？

> **観察ポイント**: 訓練データを増やすほど，訓練スコアと検証スコアの差は縮まりましたか？ 縮まらない場合、それは何を示唆しますか？（解答用コードセルに記入）

In [ ]:
# 各行の「# 説明:」に自分の言葉で意味を書いてください（コードは変更しない）
# （説明は AI に書かせず、自分で書くこと）

data = load_breast_cancer()
X, y = data.data, data.target                              # 説明:

model = DecisionTreeClassifier(random_state=0)             # 説明:

train_sizes, train_scores, valid_scores = learning_curve(  # 説明:（返り値3つは何か）
    model, X, y,
    cv=5,                                                  # 説明:（cv=5 は何をする？）
    train_sizes=np.linspace(0.1, 1.0, 10),                 # 説明:（訓練サイズをどう変える？）
)

train_mean = np.mean(train_scores, axis=1)                 # 説明:（なぜ axis=1 で平均？）
valid_mean = np.mean(valid_scores, axis=1)                 # 説明:

plt.plot(train_sizes, train_mean, "o-", label="訓練スコア")   # 説明:
plt.plot(train_sizes, valid_mean, "o-", label="検証スコア")   # 説明:
plt.xlabel("訓練データサイズ")
plt.ylabel("スコア")
plt.title("学習曲線（DecisionTree）")
plt.legend()
plt.show()

# 訓練データが最大のときの「訓練スコア - 検証スコア」の差
gap = train_mean[-1] - valid_mean[-1]                      # 説明:（この差は何を表す？）
print(f"最大サイズでの 訓練-検証 の差 = {gap:.4f}")


In [ ]:
# === ✍️ 問題1 解答（採点対象）===
# 主な提出物は上のコードセルへの # 説明: 記入。以下も記入すること。

# (1-a) `learning_curve` が返す `train_scores` と `valid_scores` はそれぞれ何を測ったスコアか
answer_1_a = """
"""

# (1-b) なぜ `axis=1` 方向に平均を取るのか
answer_1_b = """
"""

# (1-c) 訓練スコアと検証スコアの「差」が大きいことは、モデルの状態について何を意味するか
answer_1_c = """
"""

# (1-d) 観察：訓練データを増やすと差は縮まったか／縮まらない場合それは何を示唆するか
observation_1_d = """
"""



## 問題2　検証曲線によるハイパーパラメータ探索　【骨格+実験】
***

### `max_depth=None` の意味

`max_depth=None` は木の深さを制限しない（完全に成長させる）ことを意味する。決定木は訓練データを完璧に記憶できるため，`max_depth=None` では訓練スコア ≈ 1.0 になるが，テストデータでは大きく精度が落ちる（過学習の極端な例）。

### グラフの横軸について

`max_depth=[1, 2, 3, 5, 10, 20, None]` の `None` は数値ではないため，グラフ描画には工夫が必要だ。文字列ラベルとして扱うのが一般的だ：

```python
param_range = [1, 2, 3, 5, 10, 20, None]
x_labels = [str(p) for p in param_range]  # ["1", "2", "3", "5", "10", "20", "None"]
plt.xticks(range(len(x_labels)), x_labels)
```

### このグラフから何を読み取るか

検証曲線の典型的なパターン：
- `max_depth` が小さい（1〜2）: 訓練も検証も低い → 高バイアス（underfitting）
- `max_depth` が中程度（3〜5程度）: 検証スコアが最大 → 最適な複雑さ
- `max_depth` が大きい（10以上, None）: 訓練スコアは高いが検証スコアは低い → 高バリアンス（overfitting）

### 課題

下のコードセルは，`validation_curve` で `max_depth` を変化させて検証曲線を描き，各 `max_depth` での **訓練スコア・検証スコア・その差（gap）** を表に出力する **完成形**です。

`gap = 訓練スコア - 検証スコア` は **過学習の度合い**を表します（gap が大きいほど「訓練データだけに当てはまっている」）。

なお、ループ内の **CV スコアの平均を取る核心2行はあなたが書きます**（`# ★あなたが書く★`）。

実験は **2つの軸**で行ってください：
- **軸1**: `PARAM_NAME = "max_depth"` で `param_range` を **5通り以上**（例 `[1, 2, 3, 5, 10, 20, None]`）試す
- **軸2**: `PARAM_NAME = "min_samples_leaf"` に変えて `param_range = [1, 2, 5, 10, 20]` でも試す（葉の最小サンプル数も過学習に効く）

出力された各値の 訓練/検証 スコアと gap を，**✍️ 解答用コードセルの実験ログ**に記録してください。

> **観察ポイント**: `max_depth` を大きくしていくと、訓練スコアと検証スコアの **gap はどう変化**しましたか？ 検証スコアが最大になる `max_depth` はいくつでしたか？（解答用コードセルの実験ログに記録）

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target

# === ★ここを変えて実験する★：調べるパラメータ名と値の一覧（2軸） ===
# 軸1: PARAM_NAME="max_depth", param_range=[1,2,3,5,10,20,None]
# 軸2: PARAM_NAME="min_samples_leaf", param_range=[1,2,5,10,20]
PARAM_NAME = "max_depth"
param_range = [1, 2, 3, 5, 10, 20, None]

train_scores, valid_scores = validation_curve(
    DecisionTreeClassifier(random_state=0),
    X, y,
    param_name=PARAM_NAME,
    param_range=param_range,
    cv=5,
)

# ★あなたが書く★：各パラメータ値について CV5回のスコアを平均する（2行）
#   ヒント: np.mean(scores, axis=1)。axis=1 は「CV の5回」方向の平均
train_mean = ___
valid_mean = ___
x_labels = [str(p) for p in param_range]  # None を文字列として扱う

plt.plot(range(len(param_range)), train_mean, "o-", label="訓練スコア")
plt.plot(range(len(param_range)), valid_mean, "o-", label="検証スコア")
plt.xticks(range(len(param_range)), x_labels)
plt.xlabel(PARAM_NAME)
plt.ylabel("スコア")
plt.title(f"検証曲線（{PARAM_NAME}）")
plt.legend()
plt.show()

# 各パラメータ値での 訓練/検証 スコアと gap（過学習の度合い）を表示
print(f"{PARAM_NAME:>18} | {'train':>7} | {'valid':>7} | {'gap(train-valid)':>16}")
print("-" * 50)
for p, tr, va in zip(x_labels, train_mean, valid_mean):
    print(f"{p:>10} | {tr:7.4f} | {va:7.4f} | {tr - va:16.4f}")


In [ ]:
# === ✍️ 問題2 解答（採点対象）===
import pandas as pd


# (2-a) 実験ログ 軸1（`max_depth`、5通り以上）
experiment_log_axis1 = pd.DataFrame([
    {'row': 1, 'max_depth': 1, 'train_score': None, 'valid_score': None, 'gap': None},
    {'row': 2, 'max_depth': 2, 'train_score': None, 'valid_score': None, 'gap': None},
    {'row': 3, 'max_depth': 3, 'train_score': None, 'valid_score': None, 'gap': None},
    {'row': 4, 'max_depth': 5, 'train_score': None, 'valid_score': None, 'gap': None},
    {'row': 5, 'max_depth': 10, 'train_score': None, 'valid_score': None, 'gap': None},
    {'row': 6, 'max_depth': 'None', 'train_score': None, 'valid_score': None, 'gap': None},
])

# (2-b) 実験ログ 軸2（`min_samples_leaf`）
experiment_log_axis2 = pd.DataFrame([
    {'row': 1, 'min_samples_leaf': 1, 'train_score': None, 'valid_score': None, 'gap': None},
    {'row': 2, 'min_samples_leaf': 2, 'train_score': None, 'valid_score': None, 'gap': None},
    {'row': 3, 'min_samples_leaf': 5, 'train_score': None, 'valid_score': None, 'gap': None},
    {'row': 4, 'min_samples_leaf': 10, 'train_score': None, 'valid_score': None, 'gap': None},
    {'row': 5, 'min_samples_leaf': 20, 'train_score': None, 'valid_score': None, 'gap': None},
])

# (2-c) 観察：max_depth を大きくすると gap はどう変化したか／検証スコアが最大になる max_depth
observation_2_c = """
"""

# (2-d) 観察：min_samples_leaf を大きくすると過学習（gap）はどうなったか
observation_2_d = """
"""



## 問題3　過学習の診断と対処　【選択】
***

### 「訓練スコアと検証スコアの差が大きい」の意味

この状態を診断するために，問題1・2の学習曲線と検証曲線を見返そう：

- **差が大きい + 両者が収束しない**: 高バリアンス（過学習）。モデルが訓練データを「暗記」している状態
- **差が小さいが両者とも低い**: 高バイアス（過少適合）。モデルが単純すぎてパターンを学習できていない

対処法のまとめ：

| 問題 | 症状 | 対処法 |
|---|---|---|
| 過学習（高バリアンス） | 訓練高・テスト低 | 正則化，max_depth 制限，データ追加 |
| 過少適合（高バイアス） | 訓練低・テスト低 | モデルを複雑にする，特徴量を増やす |

### max_depth=3 を選ぶ根拠

問題2の検証曲線で max_depth=3 あたりが検証スコアの最大値になるはずだ。このように「検証データ（またはクロスバリデーション）でハイパーパラメータを選ぶ」プロセスが正しい方法だ。

### 課題

下のコードセルは前処理まで用意してありますが、**核心（決定木の学習と train/test 正解率の計算）はあなたが書きます**（`# ★あなたが書く★`）。実行すると，`max_depth` を制限しない決定木の **訓練正解率・テスト正解率・その差（gap）** が出力されます。これを問題1の学習曲線・問題2の検証曲線と合わせて見て，モデルの状態を **診断**してください。

> **設計判断（診断）**: いまのモデルは次のどちらの状態に近いですか？ 1つ選び、根拠（訓練スコア・テストスコア・gap の値）を解答用コードセルに書いてください。
>
> - **(状態1) 過学習（高バリアンス, high variance）** … 訓練スコアは高いが検証/テストスコアが低く、gap が大きい
> - **(状態2) 未学習（高バイアス, high bias）** … 訓練スコアもテストスコアも低い（どちらも低い）

> **設計判断（対処）**: 上で選んだ状態に対して、**効果がありそうな対処を1つ以上選び**、それぞれ「なぜ効くと思うか」を解答用コードセルに書いてください。**やみくもに全部やるのではなく、診断結果に応じて選ぶ**のが大事です。
>
> - **(A) 正則化を強める / `max_depth` を小さく制限する**
> - **(B) 学習データを増やす**
> - **(C) 特徴量を減らす**
> - **(D) モデルを単純にする**
> - **(E) モデルを複雑にする（`max_depth` を増やす・特徴量を増やす）**
> - **(F) `min_samples_leaf` を大きくする（葉に必要な最小サンプル数を増やす）**
>
> ヒント：状態1（過学習）と状態2（未学習）では、選ぶべき対処が **逆向き**になります。(E) は未学習向き、(A)(C)(D)(F) は過学習向きです。

> **考察1**: 学習曲線が「**訓練もテストも低い**」場合と、「**訓練だけ高くテストが低い**」場合とで、「データを増やすべきか」「モデルを変えるべきか」の判断はどう変わりますか？ 解答用コードセルに書いてください。


In [ ]:
# === 完成済みコード：そのまま実行し、出力（診断材料）を観察してください ===
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

# max_depth を制限しない（＝複雑な）決定木で診断する
diag_model = DecisionTreeClassifier(random_state=0)
# ★あなたが書く★：diag_model を学習し、訓練正解率 train_acc とテスト正解率 test_acc を求める（3行）
#   ヒント: .fit(X_train, y_train) で学習、.score(X, y) で正解率
___
train_acc = ___
test_acc = ___

print(f"訓練正解率   = {train_acc:.4f}")
print(f"テスト正解率 = {test_acc:.4f}")
print(f"差 gap       = {train_acc - test_acc:.4f}")
print()
print("【診断の目安】")
print(" ・訓練高 / テスト低（gap 大）   → 過学習（高バリアンス）")
print(" ・訓練低 / テスト低（どちらも低い） → 未学習（高バイアス）")


In [ ]:
# === ✍️ 問題3 解答（採点対象）===

# (3-a) 観察：訓練正解率 / テスト正解率 / gap の値
observation_3_a = """
"""

# (3-b) 設計判断（診断）：いまのモデルの状態：( 状態1 過学習 / 状態2 未学習 )
answer_3_b = """
"""

# (3-c) その根拠（上の値に触れて）
answer_3_c = """
"""

# (3-d) 設計判断（対処）：選んだ対処（A〜F から1つ以上）：(　)(　)
# 例: "B"
design3_choice = ""

# (3-e) それぞれ効く理由
answer_3_e = """
"""

# (3-f) 考察1：「訓練もテストも低い」場合と「訓練だけ高い」場合で、データを増やすべきか／モデルを変えるべきかの判断はどう変わるか
reflection1 = """
"""



## 問題4　過学習の数値的確認と次のステップ　【骨格+実験】
***

### 訓練精度とテスト精度の差で過学習を定量化

グラフで定性的に確認するだけでなく，数値で差を計算することが重要だ：

```python
gap = train_acc - test_acc
# gap が 0.05（5ポイント）以上なら過学習が疑われる
```

### GridSearchCV で自動的にハイパーパラメータを探索する方法（発展）

本回では手動で max_depth を変化させたが，実際の研究では `GridSearchCV` で自動化する：

```python
from sklearn.model_selection import GridSearchCV

param_grid = {"max_depth": [1, 2, 3, 5, 10, 20, None]}
gs = GridSearchCV(DecisionTreeClassifier(random_state=0), param_grid, cv=5)
gs.fit(X_train, y_train)
print(f"最適 max_depth: {gs.best_params_}")
print(f"最高CV スコア: {gs.best_score_:.4f}")
```

`GridSearchCV` は全組み合わせを試すが，パラメータが多い場合は `RandomizedSearchCV`（ランダムに組み合わせを選ぶ）の方が効率的だ。

### 課題

下のコードセルは，複数の `max_depth` について 訓練/テスト 正解率と gap を比較する表（DataFrame）を出力する **完成形**です。

なお、ループ内の **核心（決定木の学習と正解率の計算）はあなたが書きます**（`# ★あなたが書く★`）。`# === ★ここを変えて実験する★ ===` の **`depths_to_compare`（比較する `max_depth` の一覧）を変えて、最低5通り**実行し、結果を **✍️ 解答用コードセルの実験ログ**に記録してください。例：`[1, 2, 3, 5, 10, 20, None]`。

> **観察ポイント**: `max_depth` を大きくすると `train_acc` と `gap` はどう動きましたか？ 「gap が 0.05（5ポイント）以上なら過学習が疑われる」という目安で、どの `max_depth` が過学習していますか？（解答用コードセルに記録）

> **考察2**: `max_depth` を大きくすると訓練精度はほぼ 1.0 に近づきます（訓練データの「暗記」）。しかしテスト精度は途中から下がります。**訓練精度が高いことが、必ずしも「良いモデル」を意味しない**のはなぜか、解答用コードセルに書いてください。


In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

# === ★ここを変えて実験する★：比較する max_depth の一覧（5通り以上） ===
depths_to_compare = [1, 2, 3, 5, 10, 20, None]

rows = []
for depth in depths_to_compare:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=0)
    # ★あなたが書く★：clf を学習し、訓練正解率 train_acc とテスト正解率 test_acc を求める（3行）
    #   ヒント: .fit(X_train, y_train) と .score(X, y)
    ___
    train_acc = ___
    test_acc = ___
    rows.append({
        "max_depth": str(depth),
        "train_acc": round(train_acc, 4),
        "test_acc": round(test_acc, 4),
        "gap(train-test)": round(train_acc - test_acc, 4),
    })

result_df = pd.DataFrame(rows)
print(result_df.to_string(index=False))


In [ ]:
# === ✍️ 問題4 解答（採点対象）===
import pandas as pd


# (4-a) 実験ログ（`depths_to_compare`、5通り以上）
experiment_log = pd.DataFrame([
    {'row': 1, 'max_depth': 1, 'train_acc': None, 'test_acc': None, 'gap': None, 'overfitting': None},
    {'row': 2, 'max_depth': 2, 'train_acc': None, 'test_acc': None, 'gap': None, 'overfitting': None},
    {'row': 3, 'max_depth': 3, 'train_acc': None, 'test_acc': None, 'gap': None, 'overfitting': None},
    {'row': 4, 'max_depth': 5, 'train_acc': None, 'test_acc': None, 'gap': None, 'overfitting': None},
    {'row': 5, 'max_depth': 10, 'train_acc': None, 'test_acc': None, 'gap': None, 'overfitting': None},
    {'row': 6, 'max_depth': 'None', 'train_acc': None, 'test_acc': None, 'gap': None, 'overfitting': None},
])

# (4-b) 観察：max_depth を大きくすると train_acc と gap はどう動いたか／どの max_depth が過学習か
observation_4_b = """
"""

# (4-c) 考察2：訓練精度が高いことが必ずしも良いモデルを意味しない理由
reflection2 = """
"""

